# SpaceX Falcon 9 First Stage Landing Prediction
## Machine Learning Classification & Model Evaluation
**Author:** Dhruv Khokhriya  
**Course:** IBM Applied Data Science Capstone  

### Objectives
- Standardize the 83-dimensional feature space using `StandardScaler`.
- Partition dataset into stratified 80% train and 20% test sets.
- Train and tune 4 supervised classification models with 5-fold Stratified `GridSearchCV`:
  1. Logistic Regression
  2. Support Vector Machine (SVM)
  3. Decision Tree Classifier
  4. K-Nearest Neighbors (KNN)
- Benchmark test accuracy scores and evaluate the champion model with Confusion Matrix diagnostics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Load data
df = pd.read_csv('spacex_data.csv')
Y = df['Class'].to_numpy()

# Feature matrix with one-hot encoding
features = df[['FlightNumber', 'PayloadMass', 'Orbit', 'LaunchSite', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount']]
X = pd.get_dummies(features, columns=['Orbit', 'LaunchSite', 'LandingPad']).astype('float64').to_numpy()
print(f"Feature matrix X shape: {X.shape}, Target Y shape: {Y.shape}")

### 1. Standardize Features and Train/Test Split

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.20, random_state=42, stratify=Y)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

### 2. Logistic Regression Classifier

In [ ]:
lr = LogisticRegression(random_state=42)
lr_params = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr_cv = GridSearchCV(lr, lr_params, cv=5)
lr_cv.fit(X_train, Y_train)
lr_acc = lr_cv.score(X_test, Y_test)
print(f"Logistic Regression - Best CV Score: {lr_cv.best_score_:.4f}, Test Accuracy: {lr_acc:.4f}")

### 3. Support Vector Machine (SVM) - Champion Model

In [ ]:
svm = SVC(random_state=42)
svm_params = {'kernel': ['rbf', 'linear', 'poly', 'sigmoid'], 'C': [0.1, 1.0, 10], 'gamma': ['scale', 'auto']}
svm_cv = GridSearchCV(svm, svm_params, cv=5)
svm_cv.fit(X_train, Y_train)
svm_acc = svm_cv.score(X_test, Y_test)
print(f"SVM - Best CV Score: {svm_cv.best_score_:.4f}, Test Accuracy: {svm_acc:.4f}")

### 4. Decision Tree Classifier

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree_params = {'criterion': ['gini', 'entropy'], 'max_depth': [2, 4, 6, 8], 'min_samples_leaf': [1, 2, 4]}
tree_cv = GridSearchCV(tree, tree_params, cv=5)
tree_cv.fit(X_train, Y_train)
tree_acc = tree_cv.score(X_test, Y_test)
print(f"Decision Tree - Best CV Score: {tree_cv.best_score_:.4f}, Test Accuracy: {tree_acc:.4f}")

### 5. K-Nearest Neighbors (KNN)

In [ ]:
knn = KNeighborsClassifier()
knn_params = {'n_neighbors': [3, 5, 7, 9], 'algorithm': ['auto'], 'p': [1, 2]}
knn_cv = GridSearchCV(knn, knn_params, cv=5)
knn_cv.fit(X_train, Y_train)
knn_acc = knn_cv.score(X_test, Y_test)
print(f"KNN - Best CV Score: {knn_cv.best_score_:.4f}, Test Accuracy: {knn_acc:.4f}")

### 6. Model Accuracy Comparison Bar Chart

In [ ]:
results_df = pd.DataFrame({
    'Algorithm': ['Support Vector Machine', 'Decision Tree', 'Logistic Regression', 'K-Nearest Neighbors'],
    'Test Accuracy': [svm_acc, tree_acc, lr_acc, knn_acc],
    'CV Score': [svm_cv.best_score_, tree_cv.best_score_, lr_cv.best_score_, knn_cv.best_score_]
})
print(results_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Algorithm', y='Test Accuracy', data=results_df, palette=['#16A34A', '#2563EB', '#6366F1', '#94A3B8'])
plt.title('Machine Learning Classification Model Test Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 1.05)
plt.ylabel('Test Accuracy Score', fontsize=12)
plt.show()

### 7. Confusion Matrix Diagnostics for Best Model (SVM)

In [ ]:
y_pred = svm_cv.predict(X_test)
cm = confusion_matrix(Y_test, y_pred)
print("Classification Report:")
print(classification_report(Y_test, y_pred))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Fail', 'Predicted Success'], yticklabels=['Actual Fail', 'Actual Success'])
plt.title('SVM Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('Actual Outcome', fontsize=12)
plt.xlabel('Predicted Outcome', fontsize=12)
plt.show()